In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Factoring and Quantum Computers

This lab is designed as an exploratory example of using quantum computers for solving a difficult problems.  In this lab we will take the example of RSA encryption, example briefly how it works, and what makes it computationally difficult.  

We will explore how difficult it is factor integers using classical computers and that make RSA secure in a world absent large scale quantum computers.  

Next, we will explore the various parts that enter Shor's algorithm to use a quantum computer to factor integers.  This ranges from the basic math involved to how to contruct the next quantum gates to factor a small number.

Finally, we will use Shor's algorithm to break a very small scale RSA encryption problem, and verify that answer.


All of the answers and solutions are full given to you in the lab, most of the "questions" are really encouragement to explore and try to understand what is presented.

# First, let us explore just how hard it is to factor numbers

For this example, we will use the following routine that finds the prime factors of integers.  It works by essentially looping over all possible factors, and testing them.

In [ ]:
def factor_integer(n):
    factors = {}
    d = 2
    while n > 1:
        while n % d == 0:
            factors.setdefault(d, 0)
            factors[d] += 1
            n //= d
        d = d + 1
        if d*d > n:
            if n > 1: 
                factors[n] = 1
            break
    return factors

def find_factors_more_advanced(n):
    def pollard(n, x0=2):
        x = x0
        y = x
        d = 1
        def g(a):
            return (a * a + 1) % n
        from math import gcd
        from builtins import abs
        while d == 1:
            x = g(x)
            y = g(g(y))
            d = gcd(abs(x - y), n)
        if d == n:
            return -1
        return d
    
    factors = {}
    while n > 1:
        x0 = 2
        factor = pollard(n, x0)
        while factor < 0:
            x0 += 1
            factor = pollard(n, x0)
            print(n, factor, x0)
            if x0 > 10:
                factors.setdefault(n, 1)
                n = 1
                break
        else:
            factors.setdefault(factor, 0)
            factors[factor] += 1
            n //= factor
    return factors

Even though it is not particularly efficient, this routine's asymptotic scaling is not that different than the best algorithms, so it can serve as a good proxy to understand how things scale.  The following snippet will time how long it takes to run this routine on a set of random numbers.  We have a routine time_it that will compute time it takes to factor a number.  time_it(21) will use the above routine to factor the number 21, return a dictionary of factors with their respective powers, and the time it took to factor that number.  We then use this function in the loop to perform this measurement for many numbers with a fixed number of digits. 

In [ ]:
def time_it(value, more_than_once=False):
    from time import time_ns
    

    # run code once
    start = time_ns()
    factors = factor_integer(value)
    end = time_ns()
    if more_than_once:
        # if it took a while to run, we can just use that single run.  
        # If it ran too fast, then we need to run it many times and average them.
        n_samp = int((.1 / 20.) * 1e9 / (end - start))
        if n_samp <= 1:
            return factors, end - start
        
        start = time_ns()
        for i in range(n_samp):
            factor_integer(value)
        end = time_ns()
        return factors, (end - start) / n_samp
    return factors, (end - start)


from numpy import array, mean
from random import randrange
from time import time_ns

times = []
results = []
n_digitss = list(range(6, 12))
for n_digits in n_digitss:
    random_numbers = [randrange(10**(n_digits - 1) + 1, 10**n_digits) for i in range(100)]
    times_by_number = []
    factored = {}
    for random_number in random_numbers:
        factors, time = time_it(random_number, more_than_once=True)
        factored[random_number] = factors
        times_by_number.append(time)
    
    times.append(times_by_number)
    results.append(factored)
    print(n_digits, mean(times_by_number))

times = array(times)

Now we plot the distribution of times it took to factor numbers as a function of the number of digits.  

In [ ]:
import seaborn
ln = seaborn.violinplot(data={x:y for x, y in zip(n_digitss, times / 1e9)}, log_scale=True)
ln.axes.set_xlabel('# of Digits')
ln.axes.set_ylabel('Time to factor (seconds)')
fig = ln.get_figure()

# How many digits can we factor before this get painfully slow?

Explore using the above code snippits, how far can you go before things get painfull slow. Is there any pattern to for a given number of digits that makes some numbers really easy to factor and others really hard? How can you extrapolate to how long it would take this laptop and a python code to break 4096 bit RSA?  What about 2048 and 1024 bit RSA?  

Hint: there are approximately 3 bits for each digit in base 10, so an 8 bit number can take 3 digit to write out.

# Implementing Shor's algorithm in Qiskit!

This version of Shor's algorithm is borrowed from the very detailed intro by IBM: https://learning.quantum.ibm.com/course/fundamentals-of-quantum-algorithms/phase-estimation-and-factoring

In this lab, we only want to dig into the some of the salient features of what these sorts of algorithms look like, and how they work at a very high level. 
First we have some basic python requirements for the math and for qiskit

Next we are implementing the Oracle needed by Shor's: 

$$U\left|y\right\rangle = \left|a y \,\mathrm{mod}\, n\right\rangle$$

We will do this by treating the qubits as encoding the binary values of the number 0, 1, 2 .... n-1, and then encode relationships between the string for $y$ and $a y \,\mathrm{mod}\, n$

We will use IBM's cheesy "UnitaryGate" to cheat instead of expressing it as explict gates.  However, I want you to think about how this may be done "efficiently".

Hint: Look back at the slides.  Think about binary string of $y$ and $a y \,\mathrm{mod}\, n$ for a few choice examples of $a$, $y$, and $n$.   I would choose $a=4$ and $n=15$ to start.  Looking at smaller $n$ may not be helpful.  You can use the python **bin** function to print out the binary string for a number.

Second Hint: Swaping the values of two qubits is a "fast" operation.

Third Hint: Efficiently only means that it takes fewer than 2**n operations/gates.

In [ ]:
def shors_oracle(a, n):
    from math import gcd, floor, log
    from numpy import full
    from qiskit.circuit.library import UnitaryGate

    if gcd(a, n) > 1:
        print(
            f"Error: gcd({a},{n}) > 1"
        )  # can you explain why this condiction matter?  Perhaps look at the plot of a^m mod n if gcd(a, n) != 1, you can use the pow function in python for this.  pow(a, m, n)
    else:

        # Find the smallest number of qubits needed to store n.
        n_q = floor(log(n - 1, 2)) + 1

        U = full((2**n_q, 2**n_q), 0)

        for y in range(n):
            U[a * y % n][y] = 1

        for y in range(n, 2**n_q):
            U[y][y] = 1

        G = UnitaryGate(U)

        G.name = f"M_{a}"

        return G

Lets explore what basic gates enter the oracle we built.  Using qiskits transpile feature, we can expand our gate into basis 1 qubit gates and C nots.

What does it look like as you increase the number being factored $n$?  How does it change as you vary $a$?

In [ ]:

def draw_oracle(oracle):
    from qiskit import transpile, QuantumCircuit, QuantumRegister
    from qiskit_aer import AerSimulator


    qubits = QuantumRegister(oracle.num_qubits, name="q")
    circuit = QuantumCircuit(qubits)
    circuit.compose(oracle, qubits=list(qubits), inplace=True)
    transpile(circuit, AerSimulator()).draw(output = "mpl")

draw_oracle(shors_oracle(4, 15))

Now we will build the quantum phase estimation algorithm around our oracle gate!  Recall, for this we need powers of our gate
$$M_a, (M_a)^2, (M_a)^3, \ldots$$

We can either compute this by applying our gate over and over again, or realize that we can find a value $b$ such that
\begin{equation}
M_b = (M_a)^i
\end{equation}
for any power $i$!

In [ ]:
def order_finding_circuit(a, n):
    from builtins import pow # incase the notebook overwrote the default python pow
    from math import gcd, floor, log
    from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
    from qiskit.circuit.library import QFT
    

    if gcd(a, n) > 1:
        print(f"Error: gcd({a},{n}) > 1")
    else:
        # Find the smallest number of qubits needed to store n.
        n_q = floor(log(n - 1, 2)) + 1
        m = 2 * n_q

        control = QuantumRegister(m, name="R")
        target = QuantumRegister(n_q, name="N")
        output = ClassicalRegister(m, name="Readout")
        circuit = QuantumCircuit(control, target, output)

        # Initialize the target register to the state |1>
        circuit.x(m)

        # Add the Hadamard gates and controlled versions of the
        # multiplication gates
        for k, qubit in enumerate(control):
            circuit.h(k)
            b = pow(a, 2**k, n)

            # We will let qiskit convert our oracle gate to a controlled gate.  This comes as a penalty that scales polynomially in the number of internal gates in the oracle.
            # here the values of b are increasing by powers of a.  Instead of takeing matrix powers of our gate, we just directly built the gate that corresponds to the needed U^(2^m).
            circuit.compose(
                shors_oracle(b, n).control(),
                qubits=[qubit] + list(target),
                inplace=True,
            )

        # Apply the inverse QFT to the control register
        circuit.compose(QFT(m, inverse=True), qubits=control, inplace=True)

        # Measure the control register
        circuit.measure(control, output)

        return circuit

Now we can look at the circuit created using the draw commands in Qiskit

In [ ]:
order_finding_circuit(a=2, n=15).draw(output = "mpl")

What happens as we change the choice of $a$?  

Can you explain the numbers in $M_{b}$?

What do the qubits labeled $N_i$ represent?
What do the qubits labeled $R_i$ represent?

Why is there an X-gate on $N_0$?  Hint: what is the first step in finding the period?

Now we will simulate our circuit using the qiskit simulator.

Try setting the shots to more than 1 to see what happens?  How does this relate to dice examples earlier?

In [ ]:
def find_order(a, n, shots=1):
    # incase the notebook overwrote the default python pow
    from builtins import pow, min
    from math import gcd, floor, log
    from fractions import Fraction

    from qiskit import transpile
    from qiskit.quantum_info.operators import Operator
    from qiskit_aer import AerSimulator

    if gcd(a, n) > 1:
        print(f"Error: gcd({a},{n}) > 1")
    else:
        n_q = floor(log(n - 1, 2)) + 1
        m = 2 * n_q
        circuit = order_finding_circuit(a, n)
        transpiled_circuit = transpile(circuit, AerSimulator())

        if shots == 1:
            while True:
                result = (
                    AerSimulator()
                    .run(transpiled_circuit, shots=1, memory=True)
                    .result()
                )
                y = int(result.get_memory()[0], 2)
                r = (
                    Fraction(y / 2**m).limit_denominator(n).denominator
                )  # this is just a convenient way to convert the bit string in the ancillas to the order
                if pow(a, r, n) == 1:
                    break
            return r
        else:

            result = (
                AerSimulator()
                .run(transpiled_circuit, shots=shots, memory=True)
                .result()
            )

            orders = {}
            total_counts = 0
            for s, counts in result.get_counts().items():
                r = Fraction(int(s, 2) / 2**m).limit_denominator(n).denominator
                orders.setdefault(r, 0)
                orders[r] += counts
                total_counts += counts

            r_min = n + 1
            for r, counts in sorted(orders.items()):
                if pow(a, r, n) == 1:
                    r_min = min(r, r_min)
                    print(
                        f"Found order r={r} in {counts}/{total_counts} shots, and {a}^{r} mod {n} == 1!!"
                    )
                else:
                    print(
                        f"Found order r={r} in {counts}/{total_counts} shots, however it is a suprious answer"
                    )
            if r_min < n + 1:
                print(f"Found best choice for order is r={r_min}!")

            return r_min



find_order(4, 15, shots=1000)

Explore around as see what happen as we look at different choices for $a$, or for $n$.  Just be careful you don't take $n$ too high.

Now that we have the order, we can check if $a^{r/2} - 1$ is a factor of $n$!

In [ ]:
def find_factor_from_order(order, a, n):
    from builtins import pow # incase the notebook overwrote the default python pow
    from math import gcd
    if order % 2 == 0:
        x = pow(a, order//2, n) - 1
        d = gcd(x, n)
        if d > 1:
            print(f'Found Factor {d}!')
            return d
    print(f'Try a different a, {a} was an unlucky choice!')
    return None

find_factor_from_order(2, 7, 15)

Below is a purely classical version of how period finding can be done to find factors.

In [ ]:
def classical_shors(n):
    def _shors(n, factors):
        from math import lcm, gcd
        if n == 1:
            return factors    
        
        for a in range(2, n):
            if gcd(a, n) == 1:
                r = 1
                while pow(a, r, n) != 1:
                    r += 1
                
                if r & 1:
                    continue
        
                if (a**(r//2) + 1) % n != 0:
                    d = gcd(n, a**(r//2) + 1)
                    if d != 1:
                        return _shors(n // d, [d] + factors)
        return [n] + factors
    return _shors(n, [])

# Challenge Time
## Time to decrypt a message

You have been given my public key which consists of the numbers $17$ and $55$.  

You know that I have encrypted the number needed to exit this room with my public key and that cipher text is $47$.  

You need to break my private key, **USING SHOR's AND FACTORING BY HAND** in order to find the magic number needed leave for the day.  

If you need some help, below are two routines that can help you once you have factored my key.  Translate will take a message and translate it using either public or private key.  Extract_private_key, will take the factor you found and the public key  and compute the private key.

In [ ]:
def extract_private_key(public_key, n, factor_of_n):
    p = factor_of_n
    q = n // factor_of_n
    if p * q != n:
        print(f'{factor_of_n} is not a factor of {n}')
        return 
    
    from math import lcm
    from builtins import pow
    lam = lcm(p - 1, q - 1)
    private_key = pow(public_key, -1, lam)
    return private_key


def translate(message, key, n):
    from builtins import pow
    return pow(message, key, n)